<a href="https://colab.research.google.com/github/chetan-3028/ML/blob/main/bayes_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)

In [15]:
X, y = make_classification(
    n_samples    = 100_000,
    n_features   = 6,
    n_informative= 4,       # 4 features actually carry signal
    n_redundant  = 2,       # 2 features are noisy combinations
    n_classes    = 4,
    n_clusters_per_class = 1,
    random_state = 42
)

In [16]:
class_names   = ["Budget", "Standard", "Premium", "Elite"]
feature_names = [
    "Monthly Spend",
    "Purchase Frequency",
    "Loyalty Score",
    "Product Variety",
    "Return Rate",
    "Engagement Score"
]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\n  Class Distribution:")
for c, name in enumerate(class_names):
    count = np.sum(y == c)
    bar   = "█" * (count // 1000)
    print(f"    [{c}] {name:10s} : {count:6,} samples  {bar}")


  Class Distribution:
    [0] Budget     : 25,001 samples  █████████████████████████
    [1] Standard   : 25,013 samples  █████████████████████████
    [2] Premium    : 24,977 samples  ████████████████████████
    [3] Elite      : 25,009 samples  █████████████████████████


In [18]:
def gaussian_likelihood(x, mu, sigma_sq):
    """
    P(x | class) — probability of seeing value x
    given a Gaussian distribution with mean mu and variance sigma_sq.

    Formula:
        1                  (x - mu)²
    ─────────── × exp( − ────────── )
    √(2π × σ²)              2σ²
    """
    sigma_sq = np.maximum(sigma_sq,1e-10)
    coef = 1.0/np.sqrt(2.0*np.pi*sigma_sq)
    exponent = -((x - mu)**2)/(2.0*sigma_sq)
    return coef * np.exp(exponent)

In [23]:
class NaiveBayesClassifier:
  def fit(self,X,y):
    self.classes = np.unique(y)
    self.n_samples = len(y)
    self.n_features = len(X[0])
    self.stats = {}
    for c in self.classes:
      x_c = X[y == c]
      mu = np.mean(x_c,axis=0)
      sigma_sq = np.var(x_c,axis=0)
      prior = len(x_c)/self.n_samples
      count = len(x_c)
      self.stats[c] ={
          "mu":mu,
          "sigma_sq":sigma_sq,
          "prior":prior,
          "count":count
      }

    return self

  def _compute_log_posterior(self, x):
    """ log(x/class) +log(class) log(likelihood) + log(prior)"""
    stats = self.stats
    log_posteriors = {}
    for c in self.classes:
      log_prior = np.log(stats[c]["prior"])
      log_likelihood = np.sum(np.log(gaussian_likelihood(x, stats[c]["mu"], stats[c]["sigma_sq"])))
      log_posteriors[c] = log_prior + log_likelihood
    return log_posteriors


  def predict_single(self, x):
    log_stats = self._compute_log_posterior(x)
    predicted = max(log_stats,key = log_stats.get)
    return predicted,log_stats


  def predict(self, X):
    return np.array([self.predict_single(x)[0] for x in X])

  def predict_proba(self, X):
        """Return probability for each class (softmax of log-posteriors)."""
        probs = []
        for x in X:
            log_posts = self._compute_log_posterior(x)
            scores    = np.array([log_posts[c] for c in self.classes_])
            # Softmax: convert log-scores to probabilities
            scores   -= np.max(scores)      # numerical stability
            exp_s     = np.exp(scores)
            probs.append(exp_s / exp_s.sum())
        return np.array(probs)


In [24]:
clf = NaiveBayesClassifier()
clf.fit(X_train,y_train)



In [25]:
print("\n\n" + "=" * 65)
print("  STEP-BY-STEP CLASSIFICATION — 5 Example Points")
print("=" * 65)

# Pick 5 interesting test points
sample_indices = [0, 100, 500, 1000, 5000]

for idx in sample_indices:
    x_sample    = X_test[idx]
    true_label  = y_test[idx]
    pred, _     = clf.predict_single(x_sample)
    result      = "✓ CORRECT" if pred == true_label else "✗ WRONG"
    print(f"\n  → Predicted: {class_names[pred]:<10}  "
          f"Actual: {class_names[true_label]:<10}  {result}")




  STEP-BY-STEP CLASSIFICATION — 5 Example Points

  → Predicted: Standard    Actual: Standard    ✓ CORRECT

  → Predicted: Standard    Actual: Standard    ✓ CORRECT

  → Predicted: Elite       Actual: Elite       ✓ CORRECT

  → Predicted: Premium     Actual: Premium     ✓ CORRECT

  → Predicted: Standard    Actual: Standard    ✓ CORRECT


In [29]:
y_pred = clf.predict(X_test)


In [31]:
acc = np.mean(y_pred == y_test)
print(f"\n  Overall Accuracy: {acc:.4f} ({acc*100:.2f}%)")

print(f"\n  Confusion Matrix:")
print(f"  (Rows = Actual, Columns = Predicted)\n")
n_classes = len(class_names)
cm = np.zeros((n_classes, n_classes), dtype=int)
for true, pred in zip(y_test, y_pred):
    cm[true][pred] += 1


header = f"  {'':12}" + "".join([f"{name:>12}" for name in class_names])
print(header)
print("  " + "─" * (12 + 12 * n_classes))
for i, name in enumerate(class_names):
    row = f"  {name:<12}" + "".join([f"{cm[i][j]:>12,}" for j in range(n_classes)])
    print(row)


  Overall Accuracy: 0.7865 (78.65%)

  Confusion Matrix:
  (Rows = Actual, Columns = Predicted)

                    Budget    Standard     Premium       Elite
  ────────────────────────────────────────────────────────────
  Budget             3,907         526         432         121
  Standard             192       3,800         612         439
  Premium              225         735       3,521         490
  Elite                274          41         183       4,502


In [43]:
print(f"\n  Per-Class Performance:")
print(f"  {'Class':<12} {'TP':>8} {'FP':>8} {'FN':>8} "
      f"{'TPR':>8} {'PPV':>8} {'F1':>8} {'TNR':>8} {'NPV':>8} {'FPR':>8}")
print("\n")


for c in range(n_classes):
    TP  = cm[c][c]
    FP  = np.sum(cm[:, c]) - TP
    FN  = np.sum(cm[c, :]) - TP
    TN  = np.sum(cm) - TP - FP - FN
    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
    PPV = TP / (TP + FP) if (TP + FP) > 0 else 0
    TNR = TN / (TN + FP) if (TN + FP) > 0 else 0
    NPV = TN / (TN + FN) if (TN + FN) > 0 else 0
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    F1  = 2*TPR*PPV / (TPR+PPV) if (TPR+PPV) > 0 else 0
    print(f"  {class_names[c]:<12} {TP:>8,} {FP:>8,} {FN:>8,} "
          f"{TPR:>8.4f} {PPV:>8.4f} {F1:>8.4f} {TNR:>8.4f} {NPV:>8.4f} {FPR:>8.4f}")


  Per-Class Performance:
  Class              TP       FP       FN      TPR      PPV       F1      TNR      NPV      FPR


  Budget          3,907      691    1,079   0.7836   0.8497   0.8153   0.9540   0.9299   0.0460
  Standard        3,800    1,302    1,243   0.7535   0.7448   0.7491   0.9130   0.9166   0.0870
  Premium         3,521    1,227    1,450   0.7083   0.7416   0.7246   0.9184   0.9049   0.0816
  Elite           4,502    1,050      498   0.9004   0.8109   0.8533   0.9300   0.9655   0.0700
